# GNN vs fingerprints — one-click Colab run

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Total time ~15–20 min for the full 12-run grid on ESOL.

In [ ]:
!pip -q install rdkit lightgbm torch_geometric
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Get the code

Replace `<you>` with your GitHub username after you have pushed the repo.
While developing you can instead upload the `src/` folder to the Colab file pane.

In [ ]:
!git clone -q https://github.com/<you>/gnn-vs-fingerprints.git
%cd gnn-vs-fingerprints

## 2. Sanity checks before spending GPU time

Featuriser dimensions, one example graph, and — most importantly — proof that the
scaffold split leaks zero scaffolds from train into test.

In [ ]:
from src.data import ATOM_DIM, BOND_DIM, mol_to_graph, load_dataframe, DATASETS
from src.splits import scaffold_split, random_split, split_report

print("atom dim", ATOM_DIM, "bond dim", BOND_DIM)
print(mol_to_graph("CC(=O)Oc1ccccc1C(=O)O", -2.1))

df = load_dataframe(DATASETS["esol"])
smiles = df.smiles.tolist()
tr, va, te = scaffold_split(smiles, seed=0)
print("scaffold:", split_report(smiles, tr, va, te))
tr, va, te = random_split(smiles, seed=0)
print("random  :", split_report(smiles, tr, va, te))

The `train_test_scaffold_overlap` field is the whole point: it is 0 for the scaffold
split and a large positive number for the random split. That difference is what the rest of
the project measures.

## 3. Run the grid

{baseline, gnn} x {random, scaffold} x {seeds 0,1,2} = 12 runs.

In [ ]:
!python -m src.run_all --dataset esol --seeds 0 1 2

## 4. Figures and error analysis

In [ ]:
!python -m src.analyze --dataset esol

In [ ]:
from IPython.display import Image, Markdown, display
display(Image("results/fig_esol.png"))
display(Image("results/error_vs_sim_esol.png"))
display(Markdown(open("results/table_esol.md").read()))

In [ ]:
import pandas as pd
pd.read_csv("results/worst20_esol.csv").head(10)

## 5. Optional: a second dataset for the README

BBBP is classification (ROC-AUC), which makes the write-up stronger because the
random-vs-scaffold gap shows up on a different metric too.

In [ ]:
!python -m src.run_all --dataset bbbp --seeds 0 1 2
!python -m src.analyze --dataset bbbp

## 6. Save results back to the repo

Download `results/` and commit it, or push straight from Colab with a
[GitHub personal access token](https://github.com/settings/tokens).

In [ ]:
!zip -qr results.zip results && print("download results.zip from the file pane")